<a href="https://colab.research.google.com/github/meredithcfarley/cs-74-crisis-detection/blob/main/01preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install and import tools
!pip install contractions

import pandas as pd
import numpy as np
import re
import contractions
from sklearn.model_selection import train_test_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [contractions]


In [2]:
#load the data

from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/CS74/archive.zip")
df = df.drop(columns=["Unnamed: 0"], errors = "ignore")
print(df.shape)
df.head()

Mounted at /content/drive
(232074, 2)


,text,class
0,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,Am I weird I don't get affected by compliments...,non-suicide
2,Finally 2020 is almost over... So I can never ...,non-suicide
3,i need helpjust help me im crying so hard,suicide
4,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [3]:
#gives labels, turns true and false from dataset into 0 and 1

df["label"] = (df["class"] == 'suicide').astype(int)
print(df["label"].value_counts())
df.head()

label
1    116037
0    116037
Name: count, dtype: int64


,text,class,label
0,Ex Wife Threatening SuicideRecently I left my ...,suicide,1
1,Am I weird I don't get affected by compliments...,non-suicide,0
2,Finally 2020 is almost over... So I can never ...,non-suicide,0
3,i need helpjust help me im crying so hard,suicide,1
4,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,1


In [4]:
#separate titles from body text
def split_title(text):
    return re.sub(r"([a-z])([A-Z])", r"\1 \2", text)

print(split_title("Ex Wife Threatening SuicideRecently I left my"))
print(split_title("I'm so lostHello, my name is"))


Ex Wife Threatening Suicide Recently I left my
I'm so lost Hello, my name is


In [5]:
#full cleaning function

def clean_text(text):
    text = split_title(text)                          #separate titles (before lowercasing!)
    text = text.replace("’", "'").replace("‘", "'")   #curly apostrophes -> straight
    text = text.encode("ascii", "ignore").decode()    #drop non-standard characters
    text = contractions.fix(text)                     #"can't" -> "cannot"
    text = text.lower()                               #lowercase
    text = re.sub(r"http\S+|www\.\S+", " ", text)     #remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)             #keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()          #collapse extra spaces
    return text

#test on one real post
print("BEFORE:")
print(df["text"].iloc[0][:300])
print()
print("AFTER:")
print(clean_text(df["text"].iloc[0])[:300])



BEFORE:
Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began threatening suicide. I have tirelessly spent these paat few days talking her out of it and she keeps

AFTER:
ex wife threatening suicide recently i left my wife for good because she has cheated on me twice and lied to me so much that i have decided to refuse to go back to her as of a few days ago she began threatening suicide i have tirelessly spent these paat few days talking her out of it and she keeps h


In [6]:
#clean all of the data

df["clean_text"] = df["text"].apply(clean_text)
df["word_count"] = df["clean_text"].str.split().str.len()

print(df["word_count"].describe())
print()
print("Empty after cleaning:", (df["word_count"] == 0).sum())

count    232074.000000
mean        137.041396
std         222.429357
min           0.000000
25%          27.000000
50%          63.000000
75%         162.000000
max        9685.000000
Name: word_count, dtype: float64

Empty after cleaning: 62


In [7]:
#what are the empty rows out of curiosity
empty = df[df["word_count"] == 0]

print("Empty posts by class:")
print(empty["class"].value_counts())
print()
print("Some examples of what they originally said:")
for t in empty["text"].head(5):
    print(repr(t[:100]))

Empty posts by class:
class
non-suicide    52
suicide        10
Name: count, dtype: int64

Some examples of what they originally said:
'︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎  ︎\n︎\n︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎\n︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎ ︎'
'𝙸 𝚠𝚘𝚗𝚍𝚎𝚛 𝚑𝚘𝚠 𝚝𝚑𝚒𝚗𝚐𝚜 𝚠𝚘𝚞𝚕𝚍 𝚞𝚗𝚏𝚘𝚕𝚍 𝙹𝚘𝚎 𝚖𝚊𝚖𝚊'
'ᜎᜉᜈᜉᜇᜄᜎᜁᜐᜊᜓᜊ᜔ᜉᜎᜂᜇᜓᜌ ᜋᜎᜋᜇᜊᜌ  ᜊᜎᜊᜇᜋᜌᜅ  ᜌᜊᜇᜐᜄ ᜃᜈᜂ ᜇᜁᜉ ᜆᜇᜂ ᜃᜈᜉ ᜋ'
'Привет Здравствуйте, как вы все делаете? Я просто хочу сказать, что я умственно отсталый'
'أنا أتحدث المعكرونة هاهاها لذا كيف يمكنك جعل الكراك والميث قل لي من فضلك'


In [8]:
#drop the empty rows

df = df[df["word_count"] > 0].reset_index(drop=True)

print("New shape:", df.shape)
print(df["label"].value_counts())


New shape: (232012, 5)
label
1    116027
0    115985
Name: count, dtype: int64


In [9]:
#save cleaned data so i never have to clean again

df.to_parquet("/content/drive/MyDrive/CS74/cleaned.parquet")
print("Saved!")

Saved!


In [10]:
#train test split

X = df[["clean_text", "word_count"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)
print()
print("Train balance:")
print(y_train.value_counts(normalize=True))
print()
print("Test balance:")
print(y_test.value_counts(normalize=True))

Train: (185609, 2)  Test: (46403, 2)

Train balance:
label
1    0.500089
0    0.499911
Name: proportion, dtype: float64

Test balance:
label
1    0.500097
0    0.499903
Name: proportion, dtype: float64
